# 01 — Data Ingestion & Profiling
Lisbon Airbnb Market Intelligence | Expernetic Data Engineer Intern Assignment

Purpose: Load the raw Inside Airbnb Lisbon files, profile them (shape, types, nulls,
cardinality, duplicates, outliers), validate against basic domain rules, and produce a
data quality report that will inform our cleaning decisions in notebook 02.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

In [5]:
from pathlib import Path
print("Current working directory:", Path.cwd())
print("Contents of cwd:", list(Path.cwd().iterdir()))

Current working directory: C:\Users\User\Documents\GitHub\expernetic-data-engineer-assessment\notebooks
Contents of cwd: [WindowsPath('C:/Users/User/Documents/GitHub/expernetic-data-engineer-assessment/notebooks/.ipynb_checkpoints'), WindowsPath('C:/Users/User/Documents/GitHub/expernetic-data-engineer-assessment/notebooks/01_ingestion_profiling.ipynb')]


In [6]:
from pathlib import Path
project_root = Path.cwd().parent
data_raw = project_root / "data" / "raw"
print("Project root:", project_root)
print("data/raw exists?", data_raw.exists())
if data_raw.exists():
    print("Contents:", list(data_raw.iterdir()))
else:
    print("Contents of project root:", list(project_root.iterdir()))

Project root: C:\Users\User\Documents\GitHub\expernetic-data-engineer-assessment
data/raw exists? True
Contents: [WindowsPath('C:/Users/User/Documents/GitHub/expernetic-data-engineer-assessment/data/raw/calendar.csv.gz'), WindowsPath('C:/Users/User/Documents/GitHub/expernetic-data-engineer-assessment/data/raw/listings.csv'), WindowsPath('C:/Users/User/Documents/GitHub/expernetic-data-engineer-assessment/data/raw/neighbourhoods.csv'), WindowsPath('C:/Users/User/Documents/GitHub/expernetic-data-engineer-assessment/data/raw/neighbourhoods.geojson'), WindowsPath('C:/Users/User/Documents/GitHub/expernetic-data-engineer-assessment/data/raw/reviews.csv')]


In [7]:
listings = pd.read_csv(RAW / "listings.csv", low_memory=False)
print("listings:", listings.shape)

neighbourhoods = pd.read_csv(RAW / "neighbourhoods.csv")
print("neighbourhoods:", neighbourhoods.shape)

listings: (24876, 90)
neighbourhoods: (134, 2)


In [3]:
calendar_dtypes = {
    "listing_id": "int64",
    "available": "category",
}
calendar = pd.read_csv(
    RAW / "calendar.csv.gz", compression="gzip",
    dtype=calendar_dtypes, parse_dates=["date"]
)
print("calendar:", calendar.shape)
calendar.head(3)

calendar: (9092881, 5)


,listing_id,date,available,minimum_nights,maximum_nights
0,6499,2026-07-02,f,3,1125
1,6499,2026-07-03,f,3,1125
2,6499,2026-07-04,f,3,1125


In [8]:
reviews = pd.read_csv(
    RAW / "reviews.csv",
    parse_dates=["date"],
    dtype={"listing_id": "int64", "id": "int64", "reviewer_id": "int64"}
)
print("reviews:", reviews.shape)
reviews.head(3)

reviews: (1886932, 6)


,listing_id,id,date,reviewer_id,reviewer_name,comments
0,6499,18879225,2014-09-02,17027029,Simone,"Ola Bruno,\r<br/>\r<br/>Tive um mes Fantástico..."
1,6499,21074122,2014-10-11,7661611,Cláudio,Encontramos o apartamento de Bruno exatamente ...
2,6499,24704004,2015-01-02,20348870,Rodrigo,Estivemos em Lisboa por aproximadamente 03 (tr...


In [9]:
#schema profiling function
def profile_dataframe(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Return a per-column profiling summary: dtype, null rate, cardinality, sample value."""
    rows = []
    n = len(df)
    for col in df.columns:
        s = df[col]
        non_null = s.notna().sum()
        sample = s.dropna().iloc[0] if non_null > 0 else None
        rows.append({
            "table": name,
            "column": col,
            "dtype": str(s.dtype),
            "n_rows": n,
            "n_non_null": non_null,
            "null_rate_pct": round(100 * (n - non_null) / n, 2) if n else np.nan,
            "n_unique": s.nunique(dropna=True),
            "sample_value": str(sample)[:60] if sample is not None else None,
        })
    return pd.DataFrame(rows)

profile_listings = profile_dataframe(listings, "listings")
profile_calendar = profile_dataframe(calendar, "calendar")
profile_reviews = profile_dataframe(reviews, "reviews")
profile_neigh = profile_dataframe(neighbourhoods, "neighbourhoods")

full_profile = pd.concat([profile_listings, profile_calendar, profile_reviews, profile_neigh], ignore_index=True)
full_profile.to_csv(PROCESSED / "schema_profile.csv", index=False)
print(f"Profiled {len(full_profile)} columns across 4 tables. Saved to schema_profile.csv")
full_profile[full_profile.table == "listings"].sort_values("null_rate_pct", ascending=False).head(15)

Profiled 103 columns across 4 tables. Saved to schema_profile.csv


,table,column,dtype,n_rows,n_non_null,null_rate_pct,n_unique,sample_value
29,listings,host_total_listings_count,float64,24876,0,100.00,0,None
84,listings,instant_bookable,float64,24876,0,100.00,0,None
23,listings,host_acceptance_rate,float64,24876,0,100.00,0,None
25,listings,host_thumbnail_url,float64,24876,0,100.00,0,None
14,listings,host_since,float64,24876,0,100.00,0,None
27,listings,host_neighbourhood,float64,24876,0,100.00,0,None
22,listings,host_response_rate,float64,24876,0,100.00,0,None
30,listings,host_verifications,float64,24876,0,100.00,0,None
21,listings,host_response_time,float64,24876,0,100.00,0,None
7,listings,neighborhood_overview,float64,24876,0,100.00,0,None


In [11]:
#primary/ foreign key checks
print("listings.id unique?", listings['id'].is_unique, "| n =", listings['id'].nunique())
print("neighbourhoods (neighbourhood_group, neighbourhood) unique?",
      neighbourhoods.duplicated().sum() == 0)

orphan_calendar = (~calendar['listing_id'].isin(listings['id'])).sum()
print(f"Calendar rows with no matching listing: {orphan_calendar:,} "
      f"({100*orphan_calendar/len(calendar):.3f}%)")

orphan_reviews = (~reviews['listing_id'].isin(listings['id'])).sum()
print(f"Review rows with no matching listing: {orphan_reviews:,} "
      f"({100*orphan_reviews/len(reviews):.3f}%)")

unknown_neigh = (~listings['neighbourhood_cleansed'].isin(neighbourhoods['neighbourhood'])).sum()
print(f"Listings with neighbourhood not found in neighbourhoods.csv: {unknown_neigh:,}")

listings.id unique? True | n = 24876
neighbourhoods (neighbourhood_group, neighbourhood) unique? True
Calendar rows with no matching listing: 13,140 (0.145%)
Review rows with no matching listing: 195 (0.010%)
Listings with neighbourhood not found in neighbourhoods.csv: 0


In [12]:
#duplicate ditection
exact_dupes = listings.duplicated().sum()
id_dupes = listings['id'].duplicated().sum()
print(f"Exact duplicate rows: {exact_dupes}")
print(f"Duplicate listing IDs: {id_dupes}")

fuzzy_key = (
    listings['host_id'].astype(str) + "_" +
    listings['latitude'].round(4).astype(str) + "_" +
    listings['longitude'].round(4).astype(str) + "_" +
    listings['accommodates'].astype(str)
)
fuzzy_dupes = fuzzy_key.duplicated(keep=False).sum()
print(f"Listings sharing host + near-identical location + capacity "
      f"(possible fuzzy duplicates): {fuzzy_dupes} rows across "
      f"{fuzzy_key[fuzzy_key.duplicated(keep=False)].nunique()} groups")

Exact duplicate rows: 0
Duplicate listing IDs: 0
Listings sharing host + near-identical location + capacity (possible fuzzy duplicates): 2704 rows across 924 groups


In [13]:
#Outlier detection (IQR method)
def iqr_outlier_summary(s: pd.Series, label: str):
    s_num = pd.to_numeric(s, errors="coerce")
    q1, q3 = s_num.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((s_num < lower) | (s_num > upper)).sum()
    print(f"{label:30s} | Q1={q1:.1f} Q3={q3:.1f} IQR bounds=[{lower:.1f}, {upper:.1f}] "
          f"| outliers={n_outliers} ({100*n_outliers/s_num.notna().sum():.2f}%) "
          f"| min={s_num.min()} max={s_num.max()}")

price_numeric = listings['price'].astype(str).str.replace(r"[$,]", "", regex=True)
price_numeric = pd.to_numeric(price_numeric, errors="coerce")

iqr_outlier_summary(price_numeric, "price (raw, symbol-stripped)")
iqr_outlier_summary(listings['minimum_nights'], "minimum_nights")
iqr_outlier_summary(listings['number_of_reviews'], "number_of_reviews")
iqr_outlier_summary(listings['availability_365'], "availability_365")

price (raw, symbol-stripped)   | Q1=92.0 Q3=200.0 IQR bounds=[-70.0, 362.0] | outliers=1681 (7.43%) | min=3.67 max=16077.5
minimum_nights                 | Q1=1.0 Q3=3.0 IQR bounds=[-2.0, 6.0] | outliers=1712 (6.88%) | min=1.0 max=730.0
number_of_reviews              | Q1=3.0 Q3=96.0 IQR bounds=[-136.5, 235.5] | outliers=2409 (9.68%) | min=0 max=1931
availability_365               | Q1=132.0 Q3=323.0 IQR bounds=[-154.5, 609.5] | outliers=0 (0.00%) | min=0 max=365


In [14]:
#domain validation rules
validations = {
    "price is negative or zero": (price_numeric <= 0).sum(),
    "latitude out of Lisbon bounds (38.6-38.9)": (~listings['latitude'].between(38.6, 38.9)).sum(),
    "longitude out of Lisbon bounds (-9.5 to -9.0)": (~listings['longitude'].between(-9.5, -9.0)).sum(),
    "accommodates <= 0": (listings['accommodates'] <= 0).sum(),
    "bedrooms is null": listings['bedrooms'].isna().sum(),
    "minimum_nights > 365": (listings['minimum_nights'] > 365).sum(),
    "review score present but n_reviews == 0": (
        listings['review_scores_rating'].notna() & (listings['number_of_reviews'] == 0)
    ).sum(),
}
for rule, count in validations.items():
    pct = 100 * count / len(listings)
    print(f"{rule:55s}: {count:6d}  ({pct:.2f}%)")

price is negative or zero                              :      0  (0.00%)
latitude out of Lisbon bounds (38.6-38.9)              :   2584  (10.39%)
longitude out of Lisbon bounds (-9.5 to -9.0)          :     84  (0.34%)
accommodates <= 0                                      :      0  (0.00%)
bedrooms is null                                       :   4458  (17.92%)
minimum_nights > 365                                   :      1  (0.00%)
review score present but n_reviews == 0                :      0  (0.00%)


In [15]:
#data quality report
report_lines = []
report_lines.append("# Lisbon Airbnb — Data Quality Report\n")
report_lines.append(f"Generated from raw files scraped by Inside Airbnb (2026-07-02).\n")
report_lines.append("## Row counts\n")
report_lines.append(f"- listings: {len(listings):,}\n")
report_lines.append(f"- calendar: {len(calendar):,}\n")
report_lines.append(f"- reviews: {len(reviews):,}\n")
report_lines.append(f"- neighbourhoods: {len(neighbourhoods):,}\n")
report_lines.append("\n## Referential integrity\n")
report_lines.append(f"- Calendar rows with no matching listing: {orphan_calendar:,}\n")
report_lines.append(f"- Review rows with no matching listing: {orphan_reviews:,}\n")
report_lines.append(f"- Listings with unrecognized neighbourhood: {unknown_neigh:,}\n")
report_lines.append("\n## Duplicates\n")
report_lines.append(f"- Exact duplicate rows: {exact_dupes}\n")
report_lines.append(f"- Duplicate listing IDs: {id_dupes}\n")
report_lines.append(f"- Fuzzy duplicate candidates (same host+location+capacity): {fuzzy_dupes}\n")
report_lines.append("\n## Validation rule violations\n")
for rule, count in validations.items():
    report_lines.append(f"- {rule}: {count} ({100*count/len(listings):.2f}%)\n")

with open(PROCESSED / "data_quality_report.md", "w") as f:
    f.writelines(report_lines)

print("Data quality report saved to data/processed/data_quality_report.md")
print("".join(report_lines))

Data quality report saved to data/processed/data_quality_report.md
# Lisbon Airbnb — Data Quality Report
Generated from raw files scraped by Inside Airbnb (2026-07-02).
## Row counts
- listings: 24,876
- calendar: 9,092,881
- reviews: 1,886,932
- neighbourhoods: 134

## Referential integrity
- Calendar rows with no matching listing: 13,140
- Review rows with no matching listing: 195
- Listings with unrecognized neighbourhood: 0

## Duplicates
- Exact duplicate rows: 0
- Duplicate listing IDs: 0
- Fuzzy duplicate candidates (same host+location+capacity): 2704

## Validation rule violations
- price is negative or zero: 0 (0.00%)
- latitude out of Lisbon bounds (38.6-38.9): 2584 (10.39%)
- longitude out of Lisbon bounds (-9.5 to -9.0): 84 (0.34%)
- accommodates <= 0: 0 (0.00%)
- bedrooms is null: 4458 (17.92%)
- minimum_nights > 365: 1 (0.00%)
- review score present but n_reviews == 0: 0 (0.00%)

